# Model Definition and Evaluation
## Table of Contents
1. [Model Selection](#model-selection)
2. [Feature Engineering](#feature-engineering)
3. [Hyperparameter Tuning](#hyperparameter-tuning)
4. [Implementation](#implementation)
5. [Evaluation Metrics](#evaluation-metrics)
6. [Comparative Analysis](#comparative-analysis)


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import h5py

# (later) import models you're considering



## Model Selection

At this stage, no single model architecture has been fixed.
Instead, multiple neural network families suitable for time-series modeling are being evaluated.

We test **five model families**:
- Convolutional Neural Networks (CNN)
- Gated Recurrent Units (GRU)
- Long Short-Term Memory networks (LSTM)
- Hybrid CNN–GRU architectures
- Hybrid CNN–LSTM architectures

For each model family, **three model sizes** are explored:
- Small
- Medium
- Large

This systematic exploration allows us to analyze the impact of architectural complexity and model capacity on apnea event detection performance, while keeping the evaluation
aligned with the official challenge metric.


## Feature Engineering

No additional handcrafted features were introduced.
The only preprocessing step applied to the raw physiological signals is normalization, which helps stabilize training and ensures comparable scales across channels and recordings.


In [ ]:
# =========================
# Load input signals (X)
# =========================

input_path = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_pre files\X_normalized.h5"
with h5py.File(input_path, "r") as f:
    X = f["data"][:]  # Load dataset into NumPy array

X = X.astype("float32")
print("X shape:", X.shape)



# Perform any feature engineering steps
# Example: df['new_feature'] = df['feature1'] + df['feature2']


# =========================
# Load subject metadata
# =========================
meta = pd.read_csv(
    r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_pre files\X_train_h7ipJUo.csv"
)
print(meta.head())

subject_ids = meta["Subject_ID"].to_numpy()
print("Subjects shape:", subject_ids.shape)


# =========================
# Load target masks (y)
# =========================
y_path = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_pre files\y_train_tX9Br0C.csv"
y_df = pd.read_csv(y_path)

# Save ID separately if needed
y_id = y_df["ID"].to_numpy()

# Keep only mask columns
mask_cols = [c for c in y_df.columns if c.startswith("y_")]
y = y_df[mask_cols].to_numpy().astype("float32")

print("y shape:", y.shape)


# =========================
# Subject-wise train/validation split
# =========================
def train_val_split_by_subject(X, y, subject_ids, train_ratio=0.7, seed=42):
    rng = np.random.default_rng(seed)

    unique_subj = np.unique(subject_ids)
    rng.shuffle(unique_subj)

    n_train = int(len(unique_subj) * train_ratio)
    train_subj = unique_subj[:n_train]

    train_mask = np.isin(subject_ids, train_subj)
    val_mask   = ~train_mask

    return X[train_mask], X[val_mask], y[train_mask], y[val_mask], train_subj


X_train, X_val, y_train, y_val, train_subjects = train_val_split_by_subject(
    X, y, subject_ids, train_ratio=0.7, seed=42
)

print("Train X:", X_train.shape, "Val X:", X_val.shape)
print("Train y:", y_train.shape, "Val y:", y_val.shape)


## Hyperparameter Tuning

Hyperparameter tuning will be performed by exploring architectural variants (model family × size) and training settings.
Configurations will be compared using the official challenge metric (event-based F1) on the subject-wise validation split.

In [ ]:
# Implement hyperparameter tuning (placeholder)
# You may implement a manual search over:
# - model_family in [cnn, gru, lstm, cnn_gru, cnn_lstm]
# - model_size in [small, medium, large]
# - learning rate, batch size, etc.
# selecting the best configuration by validation event-based F1.



## Implementation

[Implement the final model(s) you've selected based on the above steps.]


In [ ]:
# Implement the final model(s)
# Example: model = YourChosenModel(best_hyperparameters)
# model.fit(X_train, y_train)

## Evaluation Metrics

Model performance is evaluated using the **event-based F1-score defined by the challenge**.

The challenge evaluates sleep apnea detection at the **event level**, rather than at the
sample or window level. Therefore, standard metrics such as accuracy or point-wise F1
are not sufficient to capture the true objective of the task.

Predictions and ground-truth labels are represented as binary masks over time.
Contiguous positive regions are converted into apnea events defined by start and end times.

Predicted and true events are matched using **Intersection-over-Union (IoU / Jaccard overlap)**:

IoU(A, B) = |A ∩ B| / (|A| + |B| − |A ∩ B|)

A predicted event is counted as a true positive if it overlaps a true event with
IoU ≥ 0.3, as specified by the challenge protocol. This evaluation strategy tolerates
small temporal misalignments while preventing double counting of events.

From the total number of true positives (TP), false positives (FP), and false negatives (FN),
Precision, Recall, and the final **event-based F1-score** are computed.

This metric is used as the primary criterion for model evaluation and comparison,
as it directly reflects the official challenge scoring.


In [ ]:
# Evaluate the model using your chosen metrics

def jaccard_overlap(events_a, events_b):
    size_a = len(events_a)
    size_b = len(events_b)

    a_start = np.array([e[0] for e in events_a])
    a_end   = np.array([e[1] for e in events_a])
    b_start = np.array([e[0] for e in events_b])
    b_end   = np.array([e[1] for e in events_b])

    a_start = np.array([a_start for _ in range(size_b)])
    a_end   = np.array([a_end for _ in range(size_b)])
    b_start = np.transpose(np.array([b_start for _ in range(size_a)]))
    b_end   = np.transpose(np.array([b_end for _ in range(size_a)]))

    len_a = a_end - a_start
    len_b = b_end - b_start

    inter_start = np.maximum(a_start, b_start)
    inter_end   = np.minimum(a_end, b_end)
    intersection = np.maximum(inter_end - inter_start, 0)

    return intersection / (len_a + len_b - intersection + 1e-12)


def extract_events_from_binary_mask(binary_mask, fs=1):
    binary_mask = np.array([0] + list(binary_mask) + [0])
    diff = np.diff(binary_mask)
    starts = np.where(diff == 1)[0] / fs
    ends   = np.where(diff == -1)[0] / fs
    return [(s, e) for s, e in zip(starts, ends)]


def compute_tp_fp_fn_for_each_entry(prediction, reference, min_iou=0.3):
    if len(prediction) == 0:
        return 0, 0, len(reference)
    if len(reference) == 0:
        return 0, len(prediction), 0

    iou = jaccard_overlap(prediction, reference)

    TP1 = np.sum(np.max(iou >= min_iou, axis=0))
    TP2 = np.sum(np.max(iou >= min_iou, axis=1))

    true_positive = min(TP1, TP2)
    false_positive = len(prediction) - true_positive
    false_negative = len(reference) - true_positive

    return true_positive, false_positive, false_negative


def event_based_f1(y_true, y_pred):
    total_tp = total_fp = total_fn = 0

    for yt, yp in zip(y_true, y_pred):
        true_events = extract_events_from_binary_mask(yt)
        pred_events = extract_events_from_binary_mask(yp)

        tp, fp, fn = compute_tp_fp_fn_for_each_entry(pred_events, true_events)
        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / (total_tp + total_fp + 1e-12)
    recall    = total_tp / (total_tp + total_fn + 1e-12)

    if precision == 0 or recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# Example usage:
# y_pred = model.predict(X_test)
# y_pred_bin = (y_pred >= 0.5).astype(int)
# f1_event = event_based_f1(y_test, y_pred_bin)
# print("Event-based F1:", f1_event)



## Comparative Analysis

At this stage, no strong baseline model is available.
As a reference, a trivial baseline that predicts no apnea events can be used to illustrate the difficulty of the task and the impact of class imbalance.

Future models will be compared using the official event-based F1-score defined by the challenge, ensuring consistency with the evaluation protocol.


In [ ]:
# Comparative Analysis code (optional)

# Dummy baseline (predict no events)
# y_pred_dummy = np.zeros_like(y_test)
# baseline_f1 = event_based_f1(y_test, y_pred_dummy)
# print(f"Dummy baseline Event-based F1: {baseline_f1:.4f}")
